In [1]:
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForVisualQuestionAnswering
import os

/data/2_data_server/cv-07/anaconda3/envs/vqa_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.1 is required but found 1.13.1+cu117
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/da

ImportError: cannot import name 'AutoProcessor' from 'transformers' (/data/2_data_server/cv-07/anaconda3/envs/vqa_env/lib/python3.9/site-packages/transformers/__init__.py)

In [ ]:
# --- 설정 ---
CSV_PATH = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/test.csv'  # 사용자님의 테스트 CSV 파일 경로
IMG_DIR = '/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/input_images'         # 이미지들이 있는 폴더 경로
HF_MODEL = "microsoft/beit_large_patch16_224_pt22k_ft22k" # BEiT 기반으로 VQA 훈련이 완료된 모델
BATCH_SIZE = 4 # GPU 메모리에 맞춰 조절


In [2]:

# --- 커스텀 데이터셋 클래스 (Hugging Face Processor 사용으로 간소화) ---
class VQADataset(Dataset):
    def __init__(self, df, img_dir):
        self.df = df
        self.img_dir = img_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.img_dir, os.path.basename(row['img_path']))
        image = Image.open(img_path).convert("RGB")
        question = row['Question']
        
        # 4개의 보기를 리스트로 만듭니다.
        options = [row['A'], row['B'], row['C'], row['D']]
        
        # 질문과 보기들을 합쳐서 모델에 넣을 텍스트 리스트를 생성
        # 예: ["질문 보기A", "질문 보기B", ...]
        texts = [f"{question} {opt}" for opt in options]

        return {"image": image, "texts": texts, "id": row['ID']}

NameError: name 'Dataset' is not defined